### modelling

In [1]:
# IMport necessary modules for modelling
import pandas as pd
 

In [2]:
df = pd.read_csv("combined_data.csv")
df.columns

 

Index(['tournament_id', 'team_id', 'team_name', 'team_code', 'stage_target',
       'matches_played', 'total_goals_scored', 'total_goals_conceded',
       'total_wins', 'total_losses', 'total_draws', 'extra_time_games',
       'penalty_shootouts', 'goal_difference', 'goals_per_match',
       'conceded_per_match', 'win_rate', 'year', 'host_country', 'count_teams',
       'is_host'],
      dtype='str')

In [ ]:
# Group  df by team_name to get historical baselines
team_profiles = df.groupby('team_name').agg({
    'goals_per_match': 'mean',
    'conceded_per_match': 'mean',
    'win_rate': 'mean'
}).reset_index()

# Rename the columns clearly
team_profiles.columns = ['team_name', 'hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']

# Load the Zindi files
zindi_train = pd.read_csv("Train.csv")
zindi_test = pd.read_csv("Test.csv")

train_data = pd.merge(zindi_train, team_profiles, left_on='country', right_on='team_name', how='left').fillna(0)
test_data = pd.merge(zindi_test, team_profiles, left_on='country', right_on='team_name', how='left').fillna(0)

#  Define features and targets using columns that now exist in train_data
feature_cols = ['hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']

X = train_data[feature_cols]
y_goals = train_data['total_goals']  # Checked: 'total_goals' matches your printout perfectly!

In [ ]:
# Group  df by team_name to get historical baselines
team_profiles = df.groupby('team_name').agg({
    'goals_per_match': 'mean',
    'conceded_per_match': 'mean',
    'win_rate': 'mean'
}).reset_index()

# Rename the columns clearly
team_profiles.columns = ['team_name', 'hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']

# 2. Load the Zindi files
zindi_train = pd.read_csv("Train.csv")
zindi_test = pd.read_csv("Test.csv")

train_data = pd.merge(zindi_train, team_profiles, left_on='country', right_on='team_name', how='left').fillna(0)
test_data = pd.merge(zindi_test, team_profiles, left_on='country', right_on='team_name', how='left').fillna(0)

#  Define features and targets using columns that now exist in train_data
feature_cols = ['hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']

X = train_data[feature_cols]
y_goals = train_data['total_goals']  # Checked: 'total_goals' matches your printout perfectly!

In [35]:
!pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 2.2 MB/s eta 0:00:00a 0:00:01


In [ ]:
import lightgbm as lgb

# Initialize the regressor with a fixed seed to guarantee reproducibility
goals_model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

# Fit the model on your historical training features
goals_model.fit(X, y_goals)

# 3. Generate predictions for the 2026 tournament teams
zindi_test['predicted_goals'] = goals_model.predict(test_data[feature_cols])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000124 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 166
[LightGBM] [Info] Number of data points in the train set: 489, number of used features: 3
[LightGBM] [Info] Start training from score 5.562372
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [37]:
# Create a baseline placeholder column for the stage reached
zindi_test['predicted_stage'] = 'group'

In [38]:
# 1. Load the official blank answer sheet
submission = pd.read_csv("SampleSubmission.csv")

# 2. Drop the placeholder columns to avoid shape/merge conflicts
submission = submission.drop(columns=['total_goals', 'Target'])

# 3. Join your predictions from the 2026 test dataframe matching on 'ID'
predictions_df = zindi_test[['ID', 'predicted_goals', 'predicted_stage']].rename(
    columns={'predicted_goals': 'total_goals', 'predicted_stage': 'Target'}
)
final_submission = pd.merge(submission, predictions_df, on='ID', how='left')

# 4. Save to CSV
final_submission.to_csv("my_first_submission.csv", index=False)
print("Success! 'my_first_submission.csv' is ready to be uploaded to Zindi.")

Success! 'my_first_submission.csv' is ready to be uploaded to Zindi.
